In [20]:
# Import required libraries
import pandas as pd
from sqlalchemy import create_engine, text, inspect, Table, MetaData
from sqlalchemy.dialects.postgresql import insert
from sqlalchemy.exc import SQLAlchemyError
import os
from dotenv import load_dotenv
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

# Define timestamps
START_TS = '2026-05-06 13:00:00'
END_TS = '2026-05-08 07:00:00'
TS_COL = 'modified_time'

print("Libraries imported successfully!")


Libraries imported successfully!


In [21]:
# Database connection configuration to SIMPEG
DB_HOST = os.getenv('DB_HOST_ASESMEN', 'localhost')
DB_PORT = os.getenv('DB_PORT_ASESMEN', '5432')  # PostgreSQL default port
DB_NAME = os.getenv('DB_DATABASE_ASESMEN', 'your_database_name')
DB_USER = os.getenv('DB_USERNAME_ASESMEN', 'your_username')
DB_PASSWORD = os.getenv('DB_PASSWORD_ASESMEN', 'your_password')

# Create connection string for PostgreSQL
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"Connecting to database: {DB_NAME} on {DB_HOST}:{DB_PORT}")
print(f"User: {DB_USER}")

# Test connection
try:
    engine_asesmen = create_engine(connection_string, echo=False)
    with engine_asesmen.connect() as connection:
        result = connection.execute(text("SELECT 1 as test"))
        print("✅ Database connection successful!")
        print(f"Connection test result: {result.fetchone()[0]}")
except SQLAlchemyError as e:
    print(f"❌ Database connection failed: {e}")
    print("Please check your database credentials in the .env file")


Connecting to database: bkd_assessment on 10.118.206.30:5432
User: bkd_assessment
✅ Database connection successful!
Connection test result: 1


In [22]:
# Database connection configuration to SIMPEG
DB_HOST = os.getenv('DB_HOST_ASESMEN_OLD', 'localhost')
DB_PORT = os.getenv('DB_PORT_ASESMEN_OLD', '5432')  # PostgreSQL default port
DB_NAME = os.getenv('DB_DATABASE_ASESMEN_OLD', 'your_database_name')
DB_USER = os.getenv('DB_USERNAME_ASESMEN_OLD', 'your_username')
DB_PASSWORD = os.getenv('DB_PASSWORD_ASESMEN_OLD', 'your_password')

# Create connection string for PostgreSQL
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"Connecting to database: {DB_NAME} on {DB_HOST}:{DB_PORT}")
print(f"User: {DB_USER}")

# Test connection
try:
    engine_asesmen_old = create_engine(connection_string, echo=False)
    with engine_asesmen_old.connect() as connection:
        result = connection.execute(text("SELECT 1 as test"))
        print("✅ Database connection successful!")
        print(f"Connection test result: {result.fetchone()[0]}")
except SQLAlchemyError as e:
    print(f"❌ Database connection failed: {e}")
    print("Please check your database credentials in the .env file")


Connecting to database: bkd_assessment on 10.110.32.122:5432
User: bkd_assessment
✅ Database connection successful!
Connection test result: 1


In [23]:
inspector = inspect(engine_asesmen_old)
tables = inspector.get_table_names()
metadata = MetaData()

print(tables)

['ccat_alat_ukur', 'AuthAssignment', 'bidang_usaha', 'AuthItem', 'AuthItemChild', 'YiiSession', 'asesor', 'backup_database', 'cache_assessment', 'ccat_analytics', 'ccat_assesment_asesor', 'ccat_assesment_batch', 'ccat_assesment_batch_jenis_ujian', 'ccat_assesment_batch_sessi', 'ccat_assesment_batch_sessi_jenis_ujian', 'ccat_assesment_batch_siska', 'ccat_assesment_merge_batch', 'ccat_assesment_peserta', 'ccat_assesment_peserta_assesment_xxx', 'ccat_assesment_batch_siska_tahapan', 'ccat_assesment_peserta_test', 'ccat_bank_soal_jawaban', 'ccat_beiw_bank_soal_pertanyaan', 'ccat_bank_soal_pertanyaan', 'ccat_beiw_asses_jawaban', 'ccat_compt_dimension_convert_rule', 'ccat_beiw_bank_soal_piljawab_mp', 'ccat_compt_convert_transfer_value', 'ccat_cpanel_leftmenu', 'ccat_critical_incident_jawaban', 'ccat_critical_incident_soal', 'ccat_disc_jawaban', 'ccat_disc_normalisasi', 'ccat_disc_parameter', 'ccat_disc_parameter_combination', 'ccat_disc_pilihan', 'ccat_disc_result', 'ccat_disc_soal', 'ccat_ep

In [24]:
for table in tables:
    try:
        columns = [col['name'] for col in inspector.get_columns(table)]
        if TS_COL not in columns:
            print(f"⏭️  {table}: no '{TS_COL}' column, skipping")
            continue

        df = pd.read_sql(
            f"SELECT * FROM {table} WHERE {TS_COL} BETWEEN '{START_TS}'::timestamp AND '{END_TS}'::timestamp",
            engine_asesmen_old
        )

        if df.empty:
            print(f"⚠️  {table}: 0 rows in range")
            continue

        # ✅ Get unique constraints in addition to PK
        pk_cols = inspector.get_pk_constraint(table)['constrained_columns']
        unique_constraints = inspector.get_unique_constraints(table)

        # Use PK by default, but fall back to first unique constraint if conflict arises
        conflict_cols = pk_cols if pk_cols else (
            unique_constraints[0]['column_names'] if unique_constraints else None
        )

        if not conflict_cols:
            print(f"⚠️  {table}: no PK or unique constraint found, skipping")
            continue

        tbl = Table(table, metadata, autoload_with=engine_asesmen, extend_existing=True)
        records = df.to_dict(orient='records')

        # ✅ Try PK first, if it fails try unique constraints
        def do_upsert(index_elements):
            stmt = insert(tbl).values(records)
            update_cols = {col: stmt.excluded[col] for col in df.columns if col not in index_elements}
            stmt = stmt.on_conflict_do_update(
                index_elements=index_elements,
                set_=update_cols
            )
            with engine_asesmen.begin() as conn:
                conn.execute(stmt)

        try:
            do_upsert(pk_cols)
        except Exception:
            if unique_constraints:
                print(f"⚠️  {table}: PK conflict failed, retrying with unique constraint...")
                do_upsert(unique_constraints[0]['column_names'])
            else:
                raise

        print(f"✅ {table}: {len(df)} rows upserted")

    except Exception as e:
        print(f"❌ {table}: {e}")

⏭️  ccat_alat_ukur: no 'modified_time' column, skipping
⏭️  AuthAssignment: no 'modified_time' column, skipping
⏭️  bidang_usaha: no 'modified_time' column, skipping
⏭️  AuthItem: no 'modified_time' column, skipping
⏭️  AuthItemChild: no 'modified_time' column, skipping
⏭️  YiiSession: no 'modified_time' column, skipping
⏭️  asesor: no 'modified_time' column, skipping
⏭️  backup_database: no 'modified_time' column, skipping
⏭️  cache_assessment: no 'modified_time' column, skipping
⏭️  ccat_analytics: no 'modified_time' column, skipping
⏭️  ccat_assesment_asesor: no 'modified_time' column, skipping
⏭️  ccat_assesment_batch: no 'modified_time' column, skipping
⏭️  ccat_assesment_batch_jenis_ujian: no 'modified_time' column, skipping
⏭️  ccat_assesment_batch_sessi: no 'modified_time' column, skipping
⏭️  ccat_assesment_batch_sessi_jenis_ujian: no 'modified_time' column, skipping
⏭️  ccat_assesment_batch_siska: no 'modified_time' column, skipping
⏭️  ccat_assesment_merge_batch: no 'modifie